In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
!nvidia-smi

Fri Jan 30 13:31:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    %pip install unsloth vllm
else:
    %pip install --upgrade -qqq uv
    !uv pip install vllm==0.11.2 unsloth-zoo unsloth
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2

In [4]:
from unsloth import FastVisionModel
import torch

model_id = 'unsloth/Qwen2.5-VL-3B-Instruct'
max_seq_length = 16384 # Must be this long for VLMs
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    # fast_inference = True, # Enable vLLM fast inference
    fast_inference = False, # Disable to fix the vLLM bug on T4
    gpu_memory_utilization = 0.8, # Reduce if out of memory
    torch_dtype = torch.float32,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 01-30 13:32:29 [fa_utils.py:64] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Qwen2_5_Vl patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.51G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [17]:
from datasets import load_dataset, Dataset

def load_hf_dataset(
    data_id,
    split='train',
    train_size = 1000,
    test_size = 0,
):
    # Use streaming
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    # Manually take train_size + test_size samples
    total_size = train_size + test_size
    sliced_data = []
    
    dataset_size = 0
    unique_cocoids = set()
    
    for example in dataset_stream:
        if dataset_size >= total_size:
            break
        
        # Ensure unique cocoids if cocoid exists
        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)
        
        sliced_data.append(example)
        dataset_size += 1

    # Convert to regular in-memory dataset
    dataset = Dataset.from_list(sliced_data)
    
    return dataset

data_id = 'jxie/coco_captions'
data_split = 'train'
data_size = 1000
dataset = load_hf_dataset(data_id, split=data_split, train_size=data_size)

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

if 'url' in dataset.features:
    dataset = dataset.map(process_image_with_url, num_proc=4) # Load and process images from URLs
else:
    dataset = dataset.map(process_image, num_proc=4) # Resize to (512, 512) and convert to RGB

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# Remove the original 'image' column
train_dataset = dataset.remove_columns('image')

# Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column('decoded_image', 'image')

In [ ]:
# # Remove bad data point with problematic vision embedding
# bad_pid = '36'
# train_dataset = train_dataset.filter(lambda x: x['pid'] != bad_pid)

In [9]:
# from vllm import SamplingParams

# sampling_params = SamplingParams(
#     temperature = 1.0,
#     top_k = 50,
#     max_tokens = 1024,
# )

# outputs = model.fast_generate(
#     {
#         "prompt": train_dataset[100]["prompt"],
#         "multi_modal_data": {"image": train_dataset[100]["image"]}
#     },
#     sampling_params,
# )
# print(outputs[0].outputs[0].text)

In [35]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_id)

In [36]:
import math
import os
from pathlib import Path
import numpy as np
from huggingface_hub import upload_file
from sklearn.preprocessing import StandardScaler

STEP = 100
train_size = len(train_dataset)
save_dir = Path(f'model_{model_id.replace("/", "_")}.data_{data_id.replace("/", "_")}_{data_split}.v2')
os.makedirs(save_dir, exist_ok=True)

scaler = StandardScaler()

for i in range(math.ceil(train_size / STEP)):
    start_idx = i * STEP
    end_idx = min(start_idx + STEP, train_size)
    range_tag = f'{start_idx}-{end_idx-1}'
    
    print("="*128)
    print("Range:", range_tag)
    print("="*128)
    
    inputs = processor.image_processor(images=list(train_dataset['image'][start_idx:end_idx]), return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)
    image_grid_thw = inputs['image_grid_thw'].to(model.device)
    
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw)

    print("Pixel values shape:", pixel_values.shape)
    print("Image grid shape:", image_grid_thw.shape)
    print("Visual embeddings shape:", visual_embs.shape)
    print()
    
    # batch = visual_embs.float() # Convert to float32 for statistics calculation -> (N, 2048)

    # # Statistics for Standardization
    # batch_n = batch.shape[0]
    # batch_sum = batch.sum(dim=0)
    # batch_sum_sq = (batch**2).sum(dim=0)

    # # Statistics for Min-Max Normalization
    # batch_min = batch.min(dim=0).values
    # batch_max = batch.max(dim=0).values

    # print("Batch size:", batch_n)
    # print("Batch sum average:", batch_sum.mean())
    # print("Batch sum squared average:", batch_sum_sq.mean())
    # print("Batch min average:", batch_min.mean())
    # print("Batch max average:", batch_max.mean())
    # print()

    # Partial fit the scaler with current batch
    scaler.partial_fit(visual_embs.detach().cpu().numpy())

    # Save vision data and statistics
    range_tag = f'{start_idx}-{end_idx-1}'
    vision_data_path = save_dir / f'{range_tag}.vision_data.npz'
    stats_path = save_dir / f'{range_tag}.stats.npz'
    
    np.savez(
        vision_data_path,
        pixel_values=pixel_values.cpu().numpy(),
        image_grid_thw=image_grid_thw.cpu().numpy(),
        visual_embs=visual_embs.cpu().numpy()
    )
    # np.savez(
    #     stats_path,
    #     batch_n=batch_n,
    #     batch_sum=batch_sum.cpu().numpy(),
    #     batch_sum_sq=batch_sum_sq.cpu().numpy(),
    #     batch_min=batch_min.cpu().numpy(),
    #     batch_max=batch_max.cpu().numpy()
    # )

    print("Vision data saved to:", vision_data_path)
    # print("Statistics saved to:", stats_path)

    upload_file(
        path_or_fileobj=str(vision_data_path),
        path_in_repo=str(vision_data_path),
        repo_id='alxxtexxr/VRBI-Dataset',
        repo_type='dataset',
    )
    # upload_file(
    #     path_or_fileobj=str(stats_path),
    #     path_in_repo=str(stats_path),
    #     repo_id='alxxtexxr/VRBI-Dataset',
    #     repo_type='dataset',
    # )

Range: 0-99
Pixel values shape: torch.Size([129600, 1176])
Image grid shape: torch.Size([100, 3])
Visual embeddings shape: torch.Size([32400, 2048])

Vision data saved to: model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/0-99.vision_data.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n.v2/0-99.vision_data.npz:   0%|          | 1.42MB /  742MB            

Range: 100-199
Pixel values shape: torch.Size([129600, 1176])
Image grid shape: torch.Size([100, 3])
Visual embeddings shape: torch.Size([32400, 2048])

Vision data saved to: model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/100-199.vision_data.npz


...

  ...2/800-899.vision_data.npz:   0%|          | 1.55MB /  742MB            

Range: 900-999
Pixel values shape: torch.Size([129600, 1176])
Image grid shape: torch.Size([100, 3])
Visual embeddings shape: torch.Size([32400, 2048])

Vision data saved to: model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/900-999.vision_data.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...2/900-999.vision_data.npz:   0%|          | 1.55MB /  742MB            

In [37]:
import joblib

# Save the scaler
scaler_path = save_dir / 'scaler.pkl'
joblib.dump(scaler, scaler_path)

print("Scaler saved to:", scaler_path)

upload_file(
    path_or_fileobj=str(scaler_path),
    path_in_repo=str(scaler_path),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)

Scaler saved to: model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/scaler.pkl


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tions_train.v2/scaler.pkl: 100%|##########| 49.8kB / 49.8kB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset/commit/6b0eef7f55764a97dcf8950763f6e12377bd9971', commit_message='Upload model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/scaler.pkl with huggingface_hub', commit_description='', oid='6b0eef7f55764a97dcf8950763f6e12377bd9971', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/VRBI-Dataset'), pr_revision=None, pr_num=None)